# Contrastive Learning on MNIST

In [ ]:
#  Install dependencies (Colab guard) 
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import sklearn
except ImportError:
    install("scikit-learn")

try:
    import umap
except ImportError:
    install("umap-learn")

try:
    import seaborn
except ImportError:
    install("seaborn")

In [ ]:
import random, math, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.datasets import MNIST

from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

#  Reproducibility 
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#  Device 
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")
print(f"PyTorch {torch.__version__}  |  Torchvision {torchvision.__version__}")

## Section 1 · Data Augmentation Module

In [ ]:
class WeakAugmentation(nn.Module):
    
    def __init__(self, img_size=28):
        super().__init__()
        self.transform = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            T.RandomAffine(degrees=5, translate=(0.05, 0.05)),
            T.ToTensor(),
            T.Normalize((0.1307,), (0.3081,)),
        ])

    def forward(self, x):
        return self.transform(x)


class StrongAugmentation(nn.Module):
    
    def __init__(self, img_size=28):
        super().__init__()
        self.transform = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.75, 1.0), ratio=(0.75, 1.33)),
            T.RandomAffine(degrees=15, translate=(0.1, 0.1), shear=5),
            T.RandomPerspective(distortion_scale=0.3, p=0.4),  
            T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.4),
            T.ToTensor(),
            T.RandomApply([T.ElasticTransform(alpha=30.0, sigma=4.0)], p=0.6),
            T.Normalize((0.1307,), (0.3081,)),
            T.RandomErasing(p=0.3, scale=(0.02, 0.20), ratio=(0.3, 3.3), value=0)
        ])

    def forward(self, x):
        return self.transform(x)


class ContrastiveDataset(Dataset):
   
    def __init__(self, mnist_dataset, augmentation):
        self.dataset = mnist_dataset
        self.aug = augmentation

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]               # PIL image + int
        view1 = self.aug.transform(img)
        view2 = self.aug.transform(img)
        return view1, view2, label


#  Visual check 
_base = MNIST(root="./data", train=True, download=True)
_weak_aug = WeakAugmentation()
_strong_aug = StrongAugmentation()

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
fig.suptitle("Augmentation Comparison  (Original | Weak x2 | Strong x2)", fontsize=13, fontweight="bold")

sample_imgs = [_base[i][0] for i in range(6)]
col_titles = ["Original", "Weak 1", "Weak 2", "Strong 1", "Strong 2", "Strong 3"]

for col, img in enumerate(sample_imgs):
    axes[0, col].imshow(np.array(img), cmap="gray")
    axes[0, col].set_title(f"digit {_base[col][1]}", fontsize=9)

for row, (aug, aug_name) in enumerate(zip([_weak_aug, _strong_aug], ["Weak", "Strong"]), start=1):
    for col, img in enumerate(sample_imgs):
        t = aug.transform(img).squeeze().numpy()
        axes[row, col].imshow(t, cmap="gray")
        if col == 0:
            axes[row, col].set_ylabel(aug_name, fontsize=10, fontweight="bold")

for ax in axes.flatten():
    ax.axis("off")
plt.tight_layout()
plt.savefig("augmentation_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Augmentation module OK")

## Section 2 · CNN Encoder

In [ ]:
class CNNEncoder(nn.Module):
    
    def __init__(self, embedding_dim: int = 128):
        super().__init__()
        self.embedding_dim = embedding_dim

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2))                                     # 28→14

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2))                                     # 14→7

        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))                             # 7→1

        self.fc = nn.Linear(128, embedding_dim)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)          # (B, 128)
        x = self.fc(x)                      # (B, embedding_dim)
        return F.normalize(x, dim=1)        # L2 norm

#  Shape test 
_enc = CNNEncoder(embedding_dim=128)
_dummy = torch.randn(4, 1, 28, 28)
_out = _enc(_dummy)
assert _out.shape == (4, 128), f"Unexpected shape: {_out.shape}"
assert torch.allclose(_out.norm(dim=1), torch.ones(4), atol=1e-5)
print(f"CNNEncoder OK  |  output shape: {_out.shape}")
print(f"Parameters: {sum(p.numel() for p in _enc.parameters()):,}")

## Section 3 · Projection Head

In [ ]:
class ProjectionHead(nn.Module):
    
    def __init__(self, input_dim: int = 128, hidden_dim: int = 256, output_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)   # also L2-normalised

_head = ProjectionHead()
_proj_out = _head(torch.randn(4, 128))
assert _proj_out.shape == (4, 128)
print(f"ProjectionHead OK  |  output shape: {_proj_out.shape}")

## Section 4 · SimCLR Model

In [ ]:
class SimCLR(nn.Module):
    
    def __init__(self, embedding_dim: int = 128, use_projection_head: bool = True):
        super().__init__()
        self.encoder = CNNEncoder(embedding_dim=embedding_dim)
        self.use_projection_head = use_projection_head
        if use_projection_head:
            self.projector = ProjectionHead(embedding_dim, 256, embedding_dim)

    def forward(self, x):
        z = self.encoder(x)                          # encoder embeddings
        if self.use_projection_head:
            p = self.projector(z)
        else:
            p = z
        return z, p

# Shape test 
_model = SimCLR(use_projection_head=True)
_dummy2 = torch.randn(4, 1, 28, 28)
_z, _p = _model(_dummy2)
assert _z.shape == _p.shape == (4, 128)
print(f"SimCLR (with head) OK  |  z: {_z.shape}  p: {_p.shape}")
total_params = sum(p.numel() for p in _model.parameters())
print(f"Total parameters: {total_params:,}")

## Section 5 · InfoNCE Loss

In [ ]:
class InfoNCELoss(nn.Module):
    
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        B = z1.size(0)
        z = torch.cat([z1, z2], dim=0)              # (2B, D)

        # Cosine similarity matrix (already normalised → just dot product)
        sim = torch.mm(z, z.T) / self.temperature   # (2B, 2B)

        # Mask out self-similarity on the diagonal
        mask = torch.eye(2 * B, device=z.device).bool()
        sim.masked_fill_(mask, -9e15)

        # Positive pairs: i↔i+B and i+B↔i
        targets = torch.cat([
            torch.arange(B, 2 * B, device=z.device),
            torch.arange(0, B,     device=z.device),
        ])                                            # (2B,)

        loss = F.cross_entropy(sim, targets)
        return loss

# Sanity check 
_loss_fn = InfoNCELoss(temperature=0.07)
_z1, _z2 = F.normalize(torch.randn(16, 128), dim=1), F.normalize(torch.randn(16, 128), dim=1)
_loss_val = _loss_fn(_z1, _z2)
assert _loss_val.item() > 0
print(f"InfoNCELoss OK  |  random-input loss: {_loss_val.item():.4f}")
print(f"Uniform-random baseline = log(2B-1) = log(31) = {math.log(31):.4f}")
print(f"A trained model should drive loss well below {math.log(31):.2f}")

## Section 6 · Training Loop

In [ ]:
#  Hyperparameters 
CONFIG = dict(
    embedding_dim   = 128,
    temperature     = 0.07,
    batch_size      = 256,
    epochs          = 50,      # increase  for richer representations
    lr              = 3e-4,
    weight_decay    = 1e-4,
    n_train_samples = 30_000,  # subset for speed; set None for full 60k
    n_eval_samples  = 2_000,
)

#  Full MNIST (PIL, no transform) 
mnist_train_raw = MNIST(root="./data", train=True,  download=True)
mnist_test_raw  = MNIST(root="./data", train=False, download=True)

if CONFIG["n_train_samples"]:
    mnist_train_raw = torch.utils.data.Subset(
        mnist_train_raw, range(CONFIG["n_train_samples"]))

print(f"Training samples : {len(mnist_train_raw):,}")
print(f"Eval samples     : {CONFIG['n_eval_samples']:,}")

In [ ]:
def build_loader(aug_class, batch_size=256):
    ds = ContrastiveDataset(mnist_train_raw, aug_class())
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=0, pin_memory=DEVICE.type=="cuda", drop_last=True)


def train_model(
    model, loader, epochs=10, lr=3e-4,
    weight_decay=1e-4, temperature=0.07, label="Model"
):
    
    model = model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = InfoNCELoss(temperature=temperature)

    loss_history = []
    model.train()

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        t0 = time.time()
        for v1, v2, _ in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            optimizer.zero_grad()
            _, p1 = model(v1)
            _, p2 = model(v2)
            loss = criterion(p1, p2)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg = epoch_loss / len(loader)
        loss_history.append(avg)
        elapsed = time.time() - t0
        print(f"  [{label}] Epoch {epoch:02d}/{epochs}  loss={avg:.4f}  ({elapsed:.1f}s)")

    return model, loss_history


def plot_loss_curves(histories: dict, title="Training Loss"):
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
    for i, (label, hist) in enumerate(histories.items()):
        ax.plot(range(1, len(hist)+1), hist, marker="o", markersize=4,
                label=label, color=colors[i % len(colors)], linewidth=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("InfoNCE Loss")
    ax.set_title(title, fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    fname = title.replace(" ", "_").lower() + ".png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.show()
    return fname


print("Training utilities defined")

## Section 7 · Evaluation — Cosine Similarity (Quantitative)

In [ ]:
def extract_embeddings(model, n_samples=2_000):
    model.eval()
    tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
    imgs, labs = [], []
    for i in range(min(n_samples, len(mnist_test_raw))):
        img, lbl = mnist_test_raw[i]
        imgs.append(tf(img))
        labs.append(lbl)
    tensor = torch.stack(imgs)          # (N, 1, 28, 28)
    all_z  = []
    with torch.no_grad():
        for start in range(0, len(tensor), 256):   # ✅ batch size 256
            z, _ = model(tensor[start:start+256].to(DEVICE))
            all_z.append(z.cpu())
    return torch.cat(all_z).numpy(), np.array(labs)


def cosine_similarity_score(embeddings, labels):
   
    n = len(labels)
    # Sample at most 2000 pairs for speed
    rng = np.random.default_rng(SEED)
    idx = rng.choice(n, size=min(n, 800), replace=False)
    emb = embeddings[idx]
    y   = labels[idx]

    S = cosine_similarity(emb)   # (M, M)

    intra, inter = [], []
    for i in range(len(y)):
        for j in range(i + 1, len(y)):
            if y[i] == y[j]:
                intra.append(S[i, j])
            else:
                inter.append(S[i, j])

    intra_sim = float(np.mean(intra))
    inter_sim = float(np.mean(inter))
    sep_score = intra_sim - inter_sim

    return {"intra_sim": intra_sim, "inter_sim": inter_sim, "sep_score": sep_score}


def plot_cosine_comparison(results: dict, title="Cosine Similarity Analysis"):
    
    labels  = list(results.keys())
    intras  = [v["intra_sim"] for v in results.values()]
    inters  = [v["inter_sim"] for v in results.values()]
    scores  = [v["sep_score"] for v in results.values()]

    x = np.arange(len(labels)); w = 0.3
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    # Left: grouped bars
    bars1 = axes[0].bar(x - w/2, intras, w, label="Intra-class", color="#4C72B0")
    bars2 = axes[0].bar(x + w/2, inters, w, label="Inter-class", color="#DD8452")
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=15, ha="right")
    axes[0].set_ylabel("Avg Cosine Similarity"); axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
    axes[0].set_title("Intra vs Inter Cosine Similarity")
    for bar in bars1: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+.005,
                                    f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
    for bar in bars2: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+.005,
                                    f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

    # Right: separation score
    colors = ["#55A868" if s > 0 else "#C44E52" for s in scores]
    bars3 = axes[1].bar(labels, scores, color=colors)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_ylabel("Separation Score (intra − inter)"); axes[1].grid(axis="y", alpha=0.3)
    axes[1].set_title("Separation Score (↑ better)")
    for bar in bars3: axes[1].text(bar.get_x()+bar.get_width()/2,
                                    bar.get_height() + (0.003 if bar.get_height() >= 0 else -0.012),
                                    f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9,
                                    fontweight="bold")
    plt.tight_layout()
    fname = title.replace(" ", "_").lower() + ".png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.show()
    return fname

print("Cosine similarity evaluation utilities defined")

## Section 8 · Evaluation — t-SNE Visualization (Qualitative)

In [ ]:
def plot_tsne(embeddings_dict: dict, labels_dict: dict, title="t-SNE Embeddings",
              n_samples=1000):
   
    n_plots = len(embeddings_dict)
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    cmap = plt.cm.get_cmap("tab10", 10)

    for ax, (name, emb) in zip(axes, embeddings_dict.items()):
        y = labels_dict[name]
        idx = np.random.choice(len(y), size=min(n_samples, len(y)), replace=False)
        emb_s, y_s = emb[idx], y[idx]

        print(f"  Running t-SNE for '{name}' on {len(idx)} samples ...", end="", flush=True)
        tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
        proj = tsne.fit_transform(emb_s)
        print(" done")

        for cls in range(10):
            mask = y_s == cls
            ax.scatter(proj[mask, 0], proj[mask, 1], c=[cmap(cls)],
                       s=12, alpha=0.7, label=str(cls))

        ax.set_title(name, fontweight="bold", fontsize=11)
        ax.axis("off")
        ax.legend(loc="upper right", markerscale=1.5, fontsize=7,
                  title="Digit", title_fontsize=8)

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    fname = title.replace(" ", "_").lower() + ".png"
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.show()
    return fname


def plot_nn_retrieval(model, n_queries=5, title="Nearest-Neighbour Retrieval"):
   
    model.eval()
    transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
    N = min(1000, len(mnist_test_raw))

    imgs_pil, ys = zip(*[mnist_test_raw[i] for i in range(N)])
    ys = np.array(ys)

    with torch.no_grad():
        embeds = []
        for img in imgs_pil:
            x = transform(img).unsqueeze(0).to(DEVICE)
            z, _ = model(x)
            embeds.append(z.cpu().squeeze())
    embeds = torch.stack(embeds).numpy()

    query_ids = [np.where(ys == cls)[0][0] for cls in range(n_queries)]
    S = cosine_similarity(embeds[query_ids], embeds)   # (n_queries, N)

    fig, axes = plt.subplots(n_queries, 6, figsize=(12, 2.2 * n_queries))
    fig.suptitle(title, fontsize=12, fontweight="bold")

    for row, (qid, sims) in enumerate(zip(query_ids, S)):
        nn_ids = np.argsort(-sims)[1:6]          # top-5 (exclude self)
        # query
        axes[row, 0].imshow(np.array(imgs_pil[qid]), cmap="gray")
        axes[row, 0].set_title(f"Query: {ys[qid]}", fontsize=8, color="red", fontweight="bold")
        axes[row, 0].axis("off")
        # neighbours
        for col, nid in enumerate(nn_ids, start=1):
            match = ys[nid] == ys[qid]
            axes[row, col].imshow(np.array(imgs_pil[nid]), cmap="gray")
            c = "green" if match else "red"
            axes[row, col].set_title(f"{ys[nid]}  sim={sims[nid]:.2f}", fontsize=7, color=c)
            axes[row, col].axis("off")

    plt.tight_layout()
    fname = title.replace(" ", "_").lower() + ".png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.show()
    return fname

print("Visualization utilities defined")

## Section 9 · Experiment 1 — With vs Without Projection Head

**Hypothesis:** The projection head absorbs augmentation-specific variance,
freeing encoder embeddings to be more generalisable.
We train two identical models; one with, one without the head.

In [ ]:
print("=" * 60)
print("EXPERIMENT 1: With vs Without Projection Head")
print("=" * 60)

loader_strong = build_loader(StrongAugmentation, CONFIG["batch_size"])

# ── Model A: with projection head ──────────────────────────────────────────
print("\n▶ Training Model A (WITH projection head) ...")
model_with_head = SimCLR(embedding_dim=CONFIG["embedding_dim"], use_projection_head=True)
model_with_head, loss_with = train_model(
    model_with_head, loader_strong,
    epochs=CONFIG["epochs"], lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    temperature=CONFIG["temperature"],
    label="W/ Head"
)

# ── Model B: without projection head ──────────────────────────────────────
print("\n▶ Training Model B (WITHOUT projection head) ...")
model_no_head = SimCLR(embedding_dim=CONFIG["embedding_dim"], use_projection_head=False)
model_no_head, loss_no = train_model(
    model_no_head, loader_strong,
    epochs=CONFIG["epochs"], lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    temperature=CONFIG["temperature"],
    label="No Head"
)

plot_loss_curves({"With Projection Head": loss_with,
                  "Without Projection Head": loss_no},
                 title="Exp1 · Training Loss: Projection Head Ablation")

In [ ]:
# ── Extract embeddings ─────────────────────────────────────────────────────
print("Extracting embeddings ...")
emb_with, lbl_with = extract_embeddings(model_with_head, CONFIG["n_eval_samples"])
emb_no,   lbl_no   = extract_embeddings(model_no_head,   CONFIG["n_eval_samples"])

# ── Score ──────────────────────────────────────────────────────────────────
score_with = cosine_similarity_score(emb_with, lbl_with)
score_no   = cosine_similarity_score(emb_no,   lbl_no)

exp1_scores = {
    "With Proj Head": score_with,
    "No Proj Head":   score_no,
}

print("\n── Experiment 1 Scores ──────────────────────────────────")
for name, s in exp1_scores.items():
    print(f"  {name:<20}  intra={s['intra_sim']:.4f}  "
          f"inter={s['inter_sim']:.4f}  sep={s['sep_score']:.4f}")

plot_cosine_comparison(exp1_scores,
    title="Exp1 · Cosine Similarity: With vs Without Projection Head")

plot_tsne({"With Proj Head": emb_with, "No Proj Head": emb_no},
          {"With Proj Head": lbl_with, "No Proj Head": lbl_no},
          title="Exp1 · t-SNE: With vs Without Projection Head")

## Section 10 · Experiment 2 — Weak vs Strong Augmentation

**Hypothesis:** Stronger augmentations create harder positive pairs, forcing the
encoder to learn more robust invariances — but too much augmentation risks losing
class information entirely.

In [ ]:
print("=" * 60)
print("EXPERIMENT 2: Weak vs Strong Augmentation")
print("=" * 60)

loader_weak   = build_loader(WeakAugmentation,   CONFIG["batch_size"])
loader_strong2 = build_loader(StrongAugmentation, CONFIG["batch_size"])

#  Model C: weak augmentation 
print("\nTraining Model C (WEAK augmentation) ...")
model_weak = SimCLR(embedding_dim=CONFIG["embedding_dim"], use_projection_head=True)
model_weak, loss_weak = train_model(
    model_weak, loader_weak,
    epochs=CONFIG["epochs"], lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    temperature=CONFIG["temperature"],
    label="Weak Aug"
)

#  Model D: strong augmentation 
print("\n Training Model D (STRONG augmentation) ...")
model_strong = SimCLR(embedding_dim=CONFIG["embedding_dim"], use_projection_head=True)
model_strong, loss_strong = train_model(
    model_strong, loader_strong2,
    epochs=CONFIG["epochs"], lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    temperature=CONFIG["temperature"],
    label="Strong Aug"
)

plot_loss_curves({"Weak Augmentation": loss_weak,
                  "Strong Augmentation": loss_strong},
                 title="Exp2 · Training Loss: Augmentation Strength")

In [ ]:
#  Extract & score 
print("Extracting embeddings ...")
emb_weak,   lbl_weak   = extract_embeddings(model_weak,   CONFIG["n_eval_samples"])
emb_strong, lbl_strong = extract_embeddings(model_strong, CONFIG["n_eval_samples"])

score_weak   = cosine_similarity_score(emb_weak,   lbl_weak)
score_strong = cosine_similarity_score(emb_strong, lbl_strong)

exp2_scores = {
    "Weak Aug":   score_weak,
    "Strong Aug": score_strong,
}

print("\n Experiment 2 Scores ")
for name, s in exp2_scores.items():
    print(f"  {name:<20}  intra={s['intra_sim']:.4f}  "
          f"inter={s['inter_sim']:.4f}  sep={s['sep_score']:.4f}")

plot_cosine_comparison(exp2_scores,
    title="Exp2 · Cosine Similarity: Weak vs Strong Augmentation")

plot_tsne({"Weak Augmentation": emb_weak, "Strong Augmentation": emb_strong},
          {"Weak Augmentation": lbl_weak, "Strong Augmentation": lbl_strong},
          title="Exp2 · t-SNE: Weak vs Strong Augmentation")

## Section 11 · Consolidated Results & Explainable Analysis

In [ ]:
#  Summary Table 
all_scores = {
    "W/ Proj Head  (strong aug)": score_with,
    "No Proj Head  (strong aug)": score_no,
    "Weak Aug   (w/ head)":       score_weak,
    "Strong Aug (w/ head)":       score_strong,
}

rows = []
for name, s in all_scores.items():
    rows.append({
        "Configuration":    name,
        "Intra-class sim":  round(s["intra_sim"], 4),
        "Inter-class sim":  round(s["inter_sim"],  4),
        "Separation Score": round(s["sep_score"],  4),
    })

df = pd.DataFrame(rows).sort_values("Separation Score", ascending=False)
df.index = range(1, len(df)+1)

print("\n" + "═"*65)
print("  CONSOLIDATED RESULTS — Separation Score (higher = better)")
print("═"*65)
print(df.to_string(index=True))
print("═"*65)
df

In [ ]:
#  4-panel annotated t-SNE grid 
configs = {
    "W/ Proj Head\n(strong aug)": (emb_with,   lbl_with),
    "No Proj Head\n(strong aug)": (emb_no,     lbl_no),
    "Weak Aug\n(w/ head)":        (emb_weak,   lbl_weak),
    "Strong Aug\n(w/ head)":      (emb_strong, lbl_strong),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Consolidated t-SNE — All Configurations", fontsize=15, fontweight="bold", y=1.01)

cmap = plt.cm.get_cmap("tab10", 10)
n_tsne = 800

for ax, (name, (emb, lbl)) in zip(axes.flatten(), configs.items()):
    idx = np.random.choice(len(lbl), size=min(n_tsne, len(lbl)), replace=False)
    print(f"  t-SNE for {name.replace(chr(10),' ')} ...", end="", flush=True)
    proj = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(emb[idx])
    print(" done")
    s = cosine_similarity_score(emb, lbl)
    for cls in range(10):
        m = lbl[idx] == cls
        ax.scatter(proj[m, 0], proj[m, 1], c=[cmap(cls)], s=12, alpha=0.7, label=str(cls))
    ax.set_title(f"{name}\nSep = {s['sep_score']:.4f}", fontsize=10, fontweight="bold")
    ax.axis("off")
    ax.legend(loc="upper right", markerscale=1.3, fontsize=6,
              title="Digit", title_fontsize=7)

plt.tight_layout()
plt.savefig("consolidated_tsne.png", dpi=130, bbox_inches="tight")
plt.show()
print("\n✅  Consolidated t-SNE saved to consolidated_tsne.png")

In [ ]:
#  Nearest-Neighbour retrieval for best model 
best_model_name = df.iloc[0]["Configuration"]
best_model = {
    "W/ Proj Head  (strong aug)": model_with_head,
    "No Proj Head  (strong aug)": model_no_head,
    "Weak Aug   (w/ head)":       model_weak,
    "Strong Aug (w/ head)":       model_strong,
}[best_model_name]

print(f"Best model: {best_model_name}")
plot_nn_retrieval(best_model, n_queries=5,
                  title=f"NN Retrieval — Best Model: {best_model_name.strip()}")